In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
import os

# --- Load the Graph and verify if preprocessing is done or not ---


try:
    df_transactions = pd.read_csv('../data/processed/train_transaction_sample.csv')
    df_identity = pd.read_csv('../data/processed/train_identity_sample.csv')
    df_merged = df_transactions.merge(df_identity, on='TransactionID', how='left')
except FileNotFoundError as e:
    print(f"Error: {e}. Please run the EDA script first.")
    exit()

df_merged['card_id'] = df_merged['card1'].astype(str) + '_' + df_merged['card2'].astype(str)
df_merged['device_id'] = df_merged['DeviceInfo'].fillna('unknown_device')

G = nx.Graph()
for _, row in df_merged.iterrows():
    txn_node = f"txn_{row['TransactionID']}"
    card_node = f"card_{row['card_id']}"
    device_node = f"device_{row['device_id']}"
    G.add_node(txn_node, type='transaction')
    G.add_node(card_node, type='card')
    G.add_node(device_node, type='device')
    G.add_edge(txn_node, card_node)
    G.add_edge(txn_node, device_node)

# --- Random Walk Generation ---
print("Generating random walks for Word2Vec...")
walk_length = 10
num_walks = 20
walks = []
for node in G.nodes():
    for _ in range(num_walks):
        walk = [node]
        for _ in range(walk_length - 1):
            neighbors = list(G.neighbors(walk[-1]))
            if neighbors:
                walk.append(np.random.choice(neighbors))
            else:
                break
        walks.append(walk)

# --- Word2Vec Model Training ---
print("Training Word2Vec model...")
embedding_size = 64
word2vec_model = Word2Vec(walks, vector_size=embedding_size, window=5, min_count=1, sg=1, workers=4)

# --- Extract and Save Transaction Embeddings ---
print("Extracting and saving GNN embeddings...")
txn_embeddings = {node: word2vec_model.wv[node] for node in G.nodes if node.startswith("txn_")}
df_embeds = pd.DataFrame.from_dict(txn_embeddings, orient="index")
df_embeds.columns = [f"gnn_embed_{i}" for i in range(df_embeds.shape[1])]

# Clean up and prepare dataframe
df_embeds.reset_index(inplace=True)
df_embeds.rename(columns={"index": "TransactionID"}, inplace=True)
df_embeds["TransactionID"] = df_embeds["TransactionID"].str.replace("txn_", "").astype(float).astype(int)

# Save the GNN embeddings 
output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)
df_embeds.to_csv(os.path.join(output_dir, 'gnn_embeddings.csv'), index=False)

print(f"\nSaved GNN embeddings to {output_dir}gnn_embeddings.csv")

Recreating graph for embedding generation...
Generating random walks for Word2Vec...


KeyboardInterrupt: 